# Data audit only: TeSent · TweetEval · CMTET · conversations.csv

This notebook **only reads and inspects** the four uploaded datasets. It does **not** label-map, merge, split, export training data, or train anything.

It ends with one verification table and a STOP. Multilingual training (`colab_final_multilingual_sentiment.ipynb`) runs only after you approve that table.

Expected layout (Google Drive):
```
MyDrive/NLP/
  telugu_tesent/      Tesent.csv, README.md
  english_tweeteval/  train_text.txt, train_labels.txt, val_text.txt, val_labels.txt,
                      test_text.txt, test_labels.txt, mapping.txt
  code_mixed_cmtet/   codemix_sentiment_data.txt, README.md
  conversations/      conversations.csv
```

In [ ]:
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/NLP")
else:
    DATA_ROOT = Path(os.environ.get("AUDIT_DATA_ROOT", r"D:\NLP\multilingual_inputs"))

TESENT_DIR, TWEETEVAL_DIR = DATA_ROOT / "telugu_tesent", DATA_ROOT / "english_tweeteval"
CMTET_DIR, CONV_DIR = DATA_ROOT / "code_mixed_cmtet", DATA_ROOT / "conversations"
AUDIT_OUT = DATA_ROOT / "audit_only" if IN_COLAB else Path(os.environ.get("AUDIT_OUT_DIR", r"D:\NLP\audit_only"))
AUDIT_OUT.mkdir(parents=True, exist_ok=True)
TARGET = {"negative", "neutral", "positive"}
MODEL_C_SKIP = "Skipped — no real conversation_id/message_order available."
print("Audit outputs ->", AUDIT_OUT)
print("Data root:", DATA_ROOT)
for d in [TESENT_DIR, TWEETEVAL_DIR, CMTET_DIR, CONV_DIR]:
    print(f"  {d.name:18s}", sorted(p.name for p in d.glob("*")) if d.exists() else "FOLDER NOT FOUND")

In [ ]:
REPORT = {}

def provenance(folder):
    f = Path(folder) / "SOURCE.json"
    if f.exists():
        m = json.loads(f.read_text(encoding="utf-8"))
        src = m.get("source", "") + (f" @ {m['commit'][:7]}" if m.get("commit") else "")
        return {"source": src, "licence_note": m.get("licence_note", "not recorded"), "provenance_file": str(f),
                "file_sha256": {k: v["sha256"] for k, v in m.get("files", {}).items()}}
    return {"source": "uploaded manually (no SOURCE.json)", "licence_note": "not recorded", "provenance_file": None}

def show_common(name, files, df, text_col, label_col, label_counts, documented_mapping, mapping_source,
                conv_cols, valid, reasons, extra=None):
    print("=" * 100); print(f"DATASET: {name}")
    print("Files:", files)
    print("Shape:", df.shape)
    print("Columns / dtypes:"); print(df.dtypes.to_string())
    print("\nSample rows:"); display(df.head(5))
    miss = df.isna().sum()
    print("Missing values:"); print(miss.to_string())
    dup = int(df.duplicated([text_col, label_col]).sum()) if text_col and label_col else None
    conflict = int((df.groupby(text_col)[label_col].nunique() > 1).sum()) if text_col and label_col else None
    print(f"Duplicate text-label pairs: {dup} | same text with different labels: {conflict}")
    print(f"Detected text column: {text_col!r} | detected label column: {label_col!r}")
    print("Original labels and counts:", label_counts)
    print(f"Documented label mapping: {documented_mapping} (source: {mapping_source})")
    print(f"conversation_id present: {conv_cols[0]} | message_order present: {conv_cols[1]}")
    if extra:
        for k, v in extra.items(): print(f"{k}: {v}")
    print(f"VALID for Negative/Neutral/Positive training: {valid}")
    for r in reasons: print("  -", r)
    REPORT[name] = {"files": files, "shape": list(df.shape), "columns": {c: str(t) for c, t in df.dtypes.items()},
                    "missing_values": {k: int(v) for k, v in miss.items()}, "duplicate_text_label_pairs": dup,
                    "same_text_conflicting_labels": conflict, "text_column": text_col, "label_column": label_col,
                    "original_label_counts": label_counts, "documented_mapping": documented_mapping,
                    "mapping_source": mapping_source, "conversation_id": conv_cols[0], "message_order": conv_cols[1],
                    "valid_for_3class_training": valid, "reasons": reasons, **(extra or {})}
    REPORT[name].update(provenance(PROV_DIR[name]))


PROV_DIR = {"TeSent (Telugu)": TESENT_DIR, "TweetEval (English)": TWEETEVAL_DIR,
            "CMTET (code-mixed)": CMTET_DIR, "conversations.csv": CONV_DIR}

def missing(name, need):
    print("=" * 100); print(f"DATASET: {name}\nNOT AUDITED, missing file(s): {need}")
    REPORT[name] = {"files": need, "valid_for_3class_training": False, "reasons": [f"missing: {need}"]}

## 1. TeSent (Telugu)

In [ ]:
f, rd = TESENT_DIR / "Tesent.csv", TESENT_DIR / "README.md"
if not f.exists():
    missing("TeSent (Telugu)", [str(f)])
else:
    df = pd.read_csv(f, encoding="utf-8")
    readme = rd.read_text(encoding="utf-8") if rd.exists() else ""
    counts = df["Label"].value_counts(dropna=False).to_dict() if "Label" in df else {}
    labels_ok = {str(k).strip().lower() for k in counts} == TARGET
    doc_ok = bool(re.search(r"Positive, Negative, Neutral", readme))
    reasons = []
    reasons.append("labels are the strings Negative/Neutral/Positive" if labels_ok else f"unexpected labels: {list(counts)}")
    reasons.append("README documents the three sentiment classes" if doc_ok else "README.md missing or does not document the classes")
    reasons.append("single sentences, no conversation_id/message_order: usable for Models A/B, not as context data")
    show_common("TeSent (Telugu)", [f.name, rd.name if rd.exists() else "README.md MISSING"], df,
                "Content" if "Content" in df else None, "Label" if "Label" in df else None,
                {str(k): int(v) for k, v in counts.items()},
                "string labels: Negative, Neutral, Positive (no numeric mapping needed)",
                "TeSent README.md: 'Sentiment Classes: Positive, Negative, Neutral'",
                ("conversation_id" in df, "message_order" in df), labels_ok and doc_ok, reasons,
                {"rows": len(df)})

## 2. TweetEval sentiment (English)

In [ ]:
need = [f"{s}_{k}.txt" for s in ["train", "val", "test"] for k in ["text", "labels"]] + ["mapping.txt"]
absent = [n for n in need if not (TWEETEVAL_DIR / n).exists()]
if absent:
    missing("TweetEval (English)", absent)
else:
    def read_lines(p):  # one item per line; empty lines are kept so text and label files stay aligned
        txt = Path(p).read_text(encoding="utf-8")
        lines = [l.rstrip("\r") for l in txt.split("\n")]
        return lines[:-1] if txt.endswith("\n") else lines
    parts, align = [], {}
    for s in ["train", "val", "test"]:
        t, l = read_lines(TWEETEVAL_DIR / f"{s}_text.txt"), read_lines(TWEETEVAL_DIR / f"{s}_labels.txt")
        align[s] = {"text_lines": len(t), "label_lines": len(l), "aligned": len(t) == len(l)}
        n = min(len(t), len(l))
        parts.append(pd.DataFrame({"text": t[:n], "label": [x.strip() for x in l[:n]], "split": s}))
    df = pd.concat(parts, ignore_index=True)
    mapping = {}
    for line in (TWEETEVAL_DIR / "mapping.txt").read_text(encoding="utf-8").splitlines():
        m = re.match(r"^\s*(\d+)\s+(\w+)\s*$", line)
        if m: mapping[m.group(1)] = m.group(2)
    counts = df["label"].value_counts().to_dict()
    reasons = []
    all_aligned = all(v["aligned"] for v in align.values())
    reasons.append("text/label files aligned in every split" if all_aligned else f"MISALIGNED files: {align}")
    map_ok = {v.lower() for v in mapping.values()} == TARGET and set(counts) <= set(mapping)
    reasons.append(f"mapping.txt documents every numeric label: {mapping}" if map_ok else f"mapping.txt incomplete/unexpected: {mapping}")
    reasons.append("tweets, no conversation_id/message_order: usable for Models A/B, not as context data")
    show_common("TweetEval (English)", need, df, "text", "label", {str(k): int(v) for k, v in counts.items()},
                {k: v.capitalize() for k, v in mapping.items()}, "TweetEval mapping.txt (cardiffnlp/tweeteval)",
                (False, False), all_aligned and map_ok, reasons,
                {"file_alignment": align, "rows_per_split": df["split"].value_counts().to_dict(),
                 "empty_texts": int((df["text"].str.strip() == "").sum())})

## 3. CMTET (Telugu-English code-mixed)

In [ ]:
SENT_RE = re.compile(r"^(NEG|NTL|POS):\s?(.*)$")
ANY_TAG_RE = re.compile(r"^([A-Z]{2,5}):\s")
LANG_TAGS = {"en", "te", "ne", "univ"}
PAPER_COUNTS = {"POS": 7925, "NEG": 7713, "NTL": 4219}

def parse_cmtet_raw(path):
    # Reads sentence tags as-is (NEG/NTL/POS). No mapping to target labels happens here.
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    recs, issues, unknown, unparsed = [], Counter(), Counter(), []
    i, n = 0, len(lines)
    while i < n:
        line = lines[i]
        if not line.strip(): i += 1; continue
        m = SENT_RE.match(line)
        if not m:
            t = ANY_TAG_RE.match(line)
            if t: unknown[t.group(1)] += 1
            issues["unparsed_line"] += 1
            if len(unparsed) < 10: unparsed.append({"line_no": i + 1, "text": line[:150]})
            i += 1; continue
        tag, sent = m.group(1), m.group(2).strip()
        j = i + 1
        while j < n and not lines[j].strip(): j += 1
        if j < n and not SENT_RE.match(lines[j]) and set(lines[j].lower().split()) <= LANG_TAGS:
            lt = lines[j].strip().lower(); nxt = j + 1
            if len(lt.split()) != len(sent.split()): issues["token_vs_language_tag_count_mismatch"] += 1
        else:
            lt = None; nxt = i + 1; issues["missing_language_tag_line"] += 1
        if not sent: issues["empty_sentence"] += 1
        recs.append({"line_no": i + 1, "text": sent, "source_label": tag, "language_tags": lt})
        i = nxt
    return pd.DataFrame(recs), dict(issues), dict(unknown), unparsed

f, rd = CMTET_DIR / "codemix_sentiment_data.txt", CMTET_DIR / "README.md"
if not f.exists():
    missing("CMTET (code-mixed)", [str(f)])
else:
    df, issues, unknown, unparsed = parse_cmtet_raw(f)
    readme = rd.read_text(encoding="utf-8") if rd.exists() else ""
    counts = df["source_label"].value_counts().to_dict()
    readme_tags = all(t in readme for t in ['"NEG"', '"NTL"', '"POS"'])
    reasons = []
    reasons.append("all sentence tags are NEG/NTL/POS" if not unknown and set(counts) <= set(PAPER_COUNTS) else f"unexpected tags: {unknown}")
    reasons.append("no unparsed lines" if not issues.get("unparsed_line") else f"{issues['unparsed_line']} unparsed lines")
    reasons.append("README.md documents NEG/NTL/POS tags" if readme_tags else "README.md missing or does not list NEG/NTL/POS")
    reasons.append("tag meaning (negative/neutral/positive) per RANLP 2021 paper Sec. 3.3; the mapping is NOT applied in this audit")
    reasons.append("sentences only, no conversation_id/message_order: usable for Models A/B, not as context data")
    valid = not unknown and not issues.get("unparsed_line") and readme_tags and set(counts) == set(PAPER_COUNTS)
    show_common("CMTET (code-mixed)", [f.name, rd.name if rd.exists() else "README.md MISSING"], df, "text", "source_label",
                {k: int(v) for k, v in counts.items()},
                {"NEG": "Negative", "NTL": "Neutral", "POS": "Positive"},
                "CMTET README.md (tags) + Kusampudi et al., RANLP 2021, Sec. 3.3 (three classes)",
                (False, False), valid, reasons,
                {"parser_issues": issues, "unknown_tags": unknown, "unparsed_examples": unparsed,
                 "paper_reported_counts": PAPER_COUNTS,
                 "difference_vs_paper": {k: counts.get(k, 0) - v for k, v in PAPER_COUNTS.items()}})

## 4. conversations.csv (manually annotated)

In [ ]:
f = CONV_DIR / "conversations.csv"
if not f.exists():
    print("=" * 100); print("DATASET: conversations.csv")
    print("Not present. No conversations are fabricated.")
    print("Model C (context-aware XLM-RoBERTa):", MODEL_C_SKIP)
    REPORT["conversations.csv"] = {"files": [], "source": "not provided", "licence_note": "n/a",
                                   "conversation_id": False, "message_order": False,
                                   "valid_for_3class_training": False, "model_c_status": MODEL_C_SKIP,
                                   "reasons": ["conversations.csv not provided", f"Model C: {MODEL_C_SKIP}"]}
else:
    df = pd.read_csv(f, encoding="utf-8")
    need = ["conversation_id", "message_order", "text", "label", "language_type"]
    absent = [c for c in need if c not in df.columns]
    reasons, extra = [], {}
    if absent:
        reasons.append(f"missing required columns: {absent}")
        counts, valid = {}, False
    else:
        counts = df["label"].value_counts(dropna=False).to_dict()
        lab_ok = {str(k).strip().lower() for k in counts if pd.notna(k)} <= TARGET
        lang_bad = sorted(set(df["language_type"].dropna()) - {"telugu", "english", "code_mixed"})
        order_num = pd.to_numeric(df["message_order"], errors="coerce")
        null_meta = int(df[["conversation_id", "message_order"]].isna().any(axis=1).sum())
        dup_order = int(df.duplicated(["conversation_id", "message_order"]).sum())
        sizes = df.groupby("conversation_id").size()
        gaps = int(sum(sorted(g.dropna().astype(int)) != list(range(1, len(g.dropna()) + 1))
                       for _, g in order_num.groupby(df["conversation_id"])))
        extra = {"n_conversations": int(df["conversation_id"].nunique()),
                 "messages_per_conversation": {"min": int(sizes.min()), "median": float(sizes.median()), "max": int(sizes.max())},
                 "conversations_with_2plus_messages": int((sizes > 1).sum()),
                 "rows_missing_conversation_id_or_message_order": null_meta,
                 "non_numeric_message_order": int(order_num.isna().sum() - df["message_order"].isna().sum()),
                 "duplicate_conversation_id_message_order": dup_order,
                 "conversations_with_order_gaps_or_not_starting_at_1": gaps,
                 "language_type_counts": df["language_type"].value_counts(dropna=False).to_dict(),
                 "invalid_language_type_values": lang_bad}
        reasons.append("labels are Negative/Neutral/Positive strings" if lab_ok else f"labels outside Negative/Neutral/Positive: {list(counts)}")
        if null_meta: reasons.append(f"{null_meta} rows missing conversation_id/message_order")
        if dup_order: reasons.append(f"{dup_order} duplicate (conversation_id, message_order) pairs")
        if extra["non_numeric_message_order"]: reasons.append("non-numeric message_order values")
        if lang_bad: reasons.append(f"invalid language_type values: {lang_bad}")
        if gaps: reasons.append(f"{gaps} conversations have order gaps / do not start at 1 (warning only)")
        if extra["n_conversations"] < 50: reasons.append(f"only {extra['n_conversations']} conversations (< 50)")
        valid = lab_ok and not null_meta and not dup_order and not lang_bad and not extra["non_numeric_message_order"] \
            and extra["n_conversations"] >= 50
        if valid: reasons.append("real conversation_id + message_order present: usable for context-aware Model C")
    show_common("conversations.csv", [f.name], df, "text" if "text" in df else None, "label" if "label" in df else None,
                {str(k): int(v) for k, v in counts.items()},
                "string labels: Negative, Neutral, Positive (no numeric mapping needed)", "your annotation guideline",
                ("conversation_id" in df, "message_order" in df), valid, reasons, extra)

## 5. Verification table, then STOP

In [ ]:
rows = []
for name, r in REPORT.items():
    ctx = bool(r.get("conversation_id")) and bool(r.get("message_order"))
    rows.append({"dataset": name, "source": r.get("source"), "licence / research-use note": r.get("licence_note"),
                 "rows": (r.get("shape") or [None])[0], "text column": r.get("text_column"), "label column": r.get("label_column"),
                 "original labels / counts": r.get("original_label_counts"), "official mapping": r.get("documented_mapping"),
                 "mapping source": r.get("mapping_source"), "dup text-label pairs": r.get("duplicate_text_label_pairs"),
                 "conversation_id": r.get("conversation_id"), "message_order": r.get("message_order"),
                 "context-aware (Model C) use": "available" if ctx and r.get("valid_for_3class_training") else
                     (r.get("model_c_status") or "no — single messages only"),
                 "valid for Neg/Neu/Pos training": r.get("valid_for_3class_training"),
                 "notes": " | ".join(r.get("reasons", []))})
table = pd.DataFrame(rows)
display(table)
table.to_csv(AUDIT_OUT / "verification_table.csv", index=False, encoding="utf-8-sig")
with open(AUDIT_OUT / "data_audit_report.json", "w", encoding="utf-8") as fh:
    json.dump(REPORT, fh, ensure_ascii=False, indent=2, default=str)
print("Saved:", AUDIT_OUT / "verification_table.csv", "and", AUDIT_OUT / "data_audit_report.json")
print("\nSTOP: audit only. No label mapping, merging, splitting, or training was performed.")
if "conversations.csv" in REPORT and REPORT["conversations.csv"].get("model_c_status"):
    print("Model C (context-aware):", MODEL_C_SKIP)
print("Review the table above and approve before any multilingual training.")